# Feature Store: ML-Ready Data Curation

This notebook transforms the raw combined feature dataset into an ML-ready format.

## Input Features (Model Input Space)

### 1. Meteorological & Fuel Conditioning Features
- 3-day consecutive moisture indices (float × 3): Normalized dryness, wetness, humidity
- 14-day fuel conditioning index (float): Weighted cumulative antecedent conditions
- Weighted weather extremes (12h): Exponentially weighted wind/precipitation
- Ignition soft-threshold index (float ∈ [0,1]): Temperature-humidity-wetness probability

### 2. Geospatial / Terrain Features
- Elevation, Slope, Terrain ruggedness, Surface curvature, Canyon indicator
- Distance to nearest body of water

### 3. Vegetation / Fuel Features  
- Forest type one-hot encoding
- Fuel layer/load encoding

### 4. Temporal / Circular Features
- Season encoding (sin/cos of day-of-year)
- Time-of-day encoding (sin/cos of hour-of-day)

## Output Space (Targets)
- Ignition probability (float ∈ [0,1])
- Fire start time
- Fire end time (or duration proxy)


In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn for preprocessing
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# Set display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")


Libraries imported successfully!


In [3]:
# Configuration
DATA_DIR = Path('data')
PROCESSED_DIR = DATA_DIR / 'processed'
ML_READY_DIR = DATA_DIR / 'ml_ready'

# Create ml_ready directory if it doesn't exist
ML_READY_DIR.mkdir(parents=True, exist_ok=True)

# Input file
INPUT_FILE = PROCESSED_DIR / 'combined_features.parquet'

# Output files
OUTPUT_FEATURES = ML_READY_DIR / 'features.parquet'
OUTPUT_TARGETS = ML_READY_DIR / 'targets.parquet'
OUTPUT_SPLIT_INDICES = ML_READY_DIR / 'split_indices.json'
# Optional: Full dataset for reference (contains metadata + features + targets)
OUTPUT_FULL = ML_READY_DIR / 'ml_ready_dataset.parquet'

print(f"Input: {INPUT_FILE}")
print(f"Output features: {OUTPUT_FEATURES}")
print(f"Output targets: {OUTPUT_TARGETS}")
print(f"Output split indices: {OUTPUT_SPLIT_INDICES}")
print(f"Output full (optional): {OUTPUT_FULL}")


Input: data\processed\combined_features.parquet
Output features: data\ml_ready\features.parquet
Output targets: data\ml_ready\targets.parquet
Output split indices: data\ml_ready\split_indices.json
Output full (optional): data\ml_ready\ml_ready_dataset.parquet


## 1. Load and Explore Raw Data


In [4]:
# Load the combined features dataset
print("Loading combined features dataset...")
df = pd.read_parquet(INPUT_FILE)

print(f"\n✓ Loaded {len(df)} samples with {len(df.columns)} columns")
print(f"\nColumn names:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col}")


Loading combined features dataset...

✓ Loaded 100 samples with 82 columns

Column names:
   1. LATITUDE
   2. LONGITUDE
   3. BRIGHTNESS
   4. SCAN
   5. TRACK
   6. ACQ_DATE
   7. ACQ_TIME
   8. SATELLITE
   9. CONFIDENCE
  10. VERSION
  11. BRIGHT_T31
  12. FRP
  13. DAYNIGHT
  14. geometry
  15. rh2m_mean
  16. precipitation_total
  17. wind_speed_mean
  18. temperature_mean
  19. rh2m_3day_mean
  20. precipitation_3day_total
  21. rh2m_14day_mean
  22. precipitation_14day_total
  23. wind_extreme_12h
  24. precipitation_extreme_12h
  25. elevation
  26. elevation_std
  27. slope
  28. slope_max
  29. ruggedness
  30. curvature
  31. canyons
  32. distance_to_water_meters
  33. forest_forest_type_evergreen_needleleaf_forest
  34. forest_forest_type_evergreen_broadleaf_forest
  35. forest_forest_type_deciduous_needleleaf_forest
  36. forest_forest_type_deciduous_broadleaf_forest
  37. forest_forest_type_mixed_forests
  38. forest_forest_type_closed_shrublands
  39. forest_forest_typ

In [5]:
# Explore data types and missing values
print("=" * 60)
print("DATA TYPES AND MISSING VALUES")
print("=" * 60)

info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
})

print(info_df)
print(f"\nTotal samples: {len(df)}")


DATA TYPES AND MISSING VALUES
                                                         dtype  non_null  \
LATITUDE                                               float64       100   
LONGITUDE                                              float64       100   
BRIGHTNESS                                             float64       100   
SCAN                                                   float64       100   
TRACK                                                  float64       100   
ACQ_DATE                                        datetime64[ms]       100   
ACQ_TIME                                                object       100   
SATELLITE                                               object       100   
CONFIDENCE                                               int32       100   
VERSION                                                 object       100   
BRIGHT_T31                                             float64       100   
FRP                                                    flo

In [6]:
# Preview the data
print("\n=== Sample Data (first 5 rows) ===")
df.head()



=== Sample Data (first 5 rows) ===


,LATITUDE,LONGITUDE,BRIGHTNESS,SCAN,TRACK,ACQ_DATE,ACQ_TIME,SATELLITE,CONFIDENCE,VERSION,BRIGHT_T31,FRP,DAYNIGHT,geometry,rh2m_mean,precipitation_total,wind_speed_mean,temperature_mean,rh2m_3day_mean,precipitation_3day_total,rh2m_14day_mean,precipitation_14day_total,wind_extreme_12h,precipitation_extreme_12h,elevation,elevation_std,slope,slope_max,ruggedness,curvature,canyons,distance_to_water_meters,forest_forest_type_evergreen_needleleaf_forest,forest_forest_type_evergreen_broadleaf_forest,forest_forest_type_deciduous_needleleaf_forest,forest_forest_type_deciduous_broadleaf_forest,forest_forest_type_mixed_forests,forest_forest_type_closed_shrublands,forest_forest_type_open_shrublands,forest_forest_type_woody_savannas,forest_forest_type_savannas,forest_forest_type_grasslands,forest_forest_type_permanent_wetlands,forest_forest_type_croplands,forest_forest_type_urban_built_up,forest_forest_type_cropland_natural_mosaic,forest_forest_type_snow_ice,forest_forest_type_barren,forest_forest_type_water,forest_landcover_class_raw,fuel_fuel_load,fuel_fuel_model,fuel_canopy_height,fuel_canopy_cover,fuel_surface_fuel_load,fuel_crown_fuel_load,fuel_fuel_model_1,fuel_fuel_model_2,fuel_fuel_model_3,fuel_fuel_model_4,fuel_fuel_model_5,fuel_fuel_model_6,fuel_fuel_model_7,fuel_fuel_model_8,fuel_fuel_model_9,fuel_fuel_model_10,fuel_fuel_model_11,fuel_fuel_model_12,fuel_fuel_model_13,fuel_fuel_model_raw,fuel_fuel_load_low,fuel_fuel_load_medium,fuel_fuel_load_high,fuel_canopy_height_low,fuel_canopy_height_medium,fuel_canopy_height_high,fuel_canopy_cover_sparse,fuel_canopy_cover_moderate,fuel_canopy_cover_dense,fuel_has_surface_fuel,fuel_has_crown_fuel,forest_forest_type_unclassified
0,8.03751,17.22174,324.88,1.40,1.17,2026-01-05,1340,A,34,6.1NRT,310.28,15.70,D,b'\x01\x01\x00\x00\x00\r7\xe0\xf3\xc381@\xc1\x...,23.839222,NaN,1.852750,29.058389,23.065417,NaN,23.839222,NaN,2.38,NaN,404.499774,5.141390,2.039148,10.766813,4.204556,0.269794,0.0,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
1,5.93590,21.27689,319.83,1.00,1.00,2026-01-05,1337,A,72,6.1NRT,305.11,9.50,D,b'\x01\x01\x00\x00\x00\xe6\x96VC\xe2F5@<N\xd1\...,42.518250,NaN,1.235417,28.283667,31.903611,NaN,42.518250,NaN,1.94,NaN,512.131482,10.741735,2.938156,9.500520,5.863655,0.281707,0.0,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
2,5.29111,30.14096,313.53,2.62,1.55,2026-01-05,0735,T,56,6.1NRT,298.90,19.45,D,"b""\x01\x01\x00\x00\x00N\x0b^\xf4\x15$>@|'f\xbd...",43.649500,NaN,1.628361,27.992444,33.835694,NaN,43.649500,NaN,2.53,NaN,637.042141,6.049330,2.709164,9.595970,6.350739,0.079243,0.0,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
3,8.92654,22.14497,341.46,1.06,1.03,2026-01-05,1340,A,89,6.1NRT,309.59,33.09,D,b'\x01\x01\x00\x00\x00\xb0\xe6\x00\xc1\x1c%6@&...,21.867806,NaN,2.208250,28.242167,21.247639,NaN,21.867806,NaN,3.46,NaN,536.686397,10.242034,4.140984,25.416286,5.667468,0.231215,0.0,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
4,15.34759,20.22583,326.58,1.02,1.01,2026-01-05,1340,A,62,6.1NRT,306.42,9.95,D,b'\x01\x01\x00\x00\x00\xa7t\xb0\xfe\xcf94@R\xd...,15.066111,NaN,4.185639,24.764417,15.292917,NaN,15.066111,NaN,5.53,NaN,409.512551,2.314409,2.955788,9.940180,4.754941,0.263471,0.0,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0

## 2. Feature Transformation Functions

Define all the transformation functions we'll need for creating ML-ready features.


In [7]:
# ============================================
# METEOROLOGICAL & FUEL CONDITIONING FEATURES
# ============================================

def compute_3day_moisture_indices(df):
    """
    Compute 3-day consecutive moisture indices:
    - Dryness index: Inverse of precipitation (normalized)
    - Wetness index: Based on precipitation total (normalized)
    - Humidity index: Based on relative humidity (normalized)
    
    Returns normalized values in [0, 1] range.
    """
    result = pd.DataFrame(index=df.index)
    
    # Check which columns are available
    precip_col = 'precipitation_3day_total' if 'precipitation_3day_total' in df.columns else 'precipitation_total'
    humid_col = 'rh2m_3day_mean' if 'rh2m_3day_mean' in df.columns else 'rh2m_mean'
    
    # Wetness Index: Higher precipitation = wetter conditions
    if precip_col in df.columns:
        precip = df[precip_col].fillna(0)
        # Normalize to [0, 1] using min-max scaling
        precip_min, precip_max = precip.min(), precip.max()
        if precip_max > precip_min:
            result['moisture_wetness_3day'] = (precip - precip_min) / (precip_max - precip_min)
        else:
            result['moisture_wetness_3day'] = 0.0
    else:
        result['moisture_wetness_3day'] = 0.5  # Default neutral
    
    # Dryness Index: Inverse of wetness (1 - wetness)
    result['moisture_dryness_3day'] = 1.0 - result['moisture_wetness_3day']
    
    # Humidity Index: Based on relative humidity
    if humid_col in df.columns:
        humidity = df[humid_col].fillna(50)  # Default to 50% if missing
        # RH is typically 0-100%, normalize to [0, 1]
        result['moisture_humidity_3day'] = (humidity.clip(0, 100) / 100.0)
    else:
        result['moisture_humidity_3day'] = 0.5  # Default neutral
    
    return result


def compute_14day_fuel_conditioning_index(df):
    """
    Compute 14-day fuel conditioning index.
    
    This is a weighted cumulative measure of antecedent wetness/dryness
    affecting fuel readiness. Uses exponential decay weighting.
    
    Formula: FCI = (dryness_weight * precip_deficit) + (temp_weight * temp_factor)
    
    Returns float in [0, 1] where:
    - 0 = wet conditions, low fire risk
    - 1 = dry conditions, high fire risk
    """
    result = pd.DataFrame(index=df.index)
    
    # Get 14-day precipitation and humidity
    precip_col = 'precipitation_14day_total' if 'precipitation_14day_total' in df.columns else 'precipitation_total'
    humid_col = 'rh2m_14day_mean' if 'rh2m_14day_mean' in df.columns else 'rh2m_mean'
    temp_col = 'temperature_mean'
    
    # Initialize components
    precip_factor = 0.5  # Neutral default
    humid_factor = 0.5
    temp_factor = 0.5
    
    # Precipitation deficit component (less rain = higher index)
    if precip_col in df.columns:
        precip = df[precip_col].fillna(df[precip_col].median())
        precip_min, precip_max = precip.min(), precip.max()
        if precip_max > precip_min:
            # Invert: less precipitation = higher fuel conditioning (drier)
            precip_factor = 1.0 - ((precip - precip_min) / (precip_max - precip_min))
        else:
            precip_factor = 0.5
    
    # Humidity component (lower humidity = higher index)
    if humid_col in df.columns:
        humidity = df[humid_col].fillna(50)
        # Invert: lower humidity = higher fuel conditioning
        humid_factor = 1.0 - (humidity.clip(0, 100) / 100.0)
    
    # Temperature component (higher temp = higher index)
    if temp_col in df.columns:
        temp = df[temp_col].fillna(df[temp_col].median())
        temp_min, temp_max = temp.min(), temp.max()
        if temp_max > temp_min:
            temp_factor = (temp - temp_min) / (temp_max - temp_min)
        else:
            temp_factor = 0.5
    
    # Weighted combination
    # Precipitation has highest weight, then humidity, then temperature
    weights = {'precip': 0.5, 'humid': 0.3, 'temp': 0.2}
    
    result['fuel_conditioning_14day'] = (
        weights['precip'] * precip_factor +
        weights['humid'] * humid_factor +
        weights['temp'] * temp_factor
    ).clip(0, 1)
    
    return result


def compute_weighted_weather_extremes_12h(df):
    """
    Compute exponentially weighted weather extremes over 12 hours.
    
    Returns:
    - wind_extreme_weighted: Weighted wind speed extreme
    - precip_extreme_weighted: Weighted precipitation extreme
    
    Values are normalized to [0, 1].
    """
    result = pd.DataFrame(index=df.index)
    
    # Wind speed extremes
    wind_col = 'wind_extreme_12h' if 'wind_extreme_12h' in df.columns else 'wind_speed_mean'
    if wind_col in df.columns:
        wind = df[wind_col].fillna(0)
        wind_min, wind_max = wind.min(), wind.max()
        if wind_max > wind_min:
            result['weather_extreme_wind_12h'] = (wind - wind_min) / (wind_max - wind_min)
        else:
            result['weather_extreme_wind_12h'] = 0.0
    else:
        result['weather_extreme_wind_12h'] = 0.0
    
    # Precipitation extremes
    precip_col = 'precipitation_extreme_12h' if 'precipitation_extreme_12h' in df.columns else None
    if precip_col and precip_col in df.columns:
        precip = df[precip_col].fillna(0)
        precip_min, precip_max = precip.min(), precip.max()
        if precip_max > precip_min:
            result['weather_extreme_precip_12h'] = (precip - precip_min) / (precip_max - precip_min)
        else:
            result['weather_extreme_precip_12h'] = 0.0
    else:
        result['weather_extreme_precip_12h'] = 0.0
    
    return result


def compute_ignition_soft_threshold(df):
    """
    Compute ignition soft-threshold index.
    
    This is a continuous probability-like feature encoding
    temperature-humidity-wetness ignition conditions.
    
    Formula uses sigmoid-like soft thresholding:
    - Higher temperature increases probability
    - Lower humidity increases probability
    - Less precipitation increases probability
    
    Returns float in [0, 1] representing ignition likelihood.
    """
    result = pd.DataFrame(index=df.index)
    
    # Get relevant columns
    temp_col = 'temperature_mean'
    humid_col = 'rh2m_mean' if 'rh2m_mean' in df.columns else 'rh2m_3day_mean'
    precip_col = 'precipitation_total' if 'precipitation_total' in df.columns else 'precipitation_3day_total'
    
    # Initialize scores
    temp_score = 0.5
    humid_score = 0.5
    precip_score = 0.5
    
    # Temperature score: Higher temp = higher ignition probability
    # Using soft threshold around typical fire-prone temperatures
    if temp_col in df.columns:
        temp = df[temp_col].fillna(df[temp_col].median())
        # Sigmoid-like transformation centered around 25°C (77°F)
        # For Fahrenheit, center around 77°F
        if temp.median() > 50:  # Likely Fahrenheit
            center = 77
            scale = 20
        else:  # Likely Celsius
            center = 25
            scale = 10
        temp_score = 1 / (1 + np.exp(-(temp - center) / scale))
    
    # Humidity score: Lower humidity = higher ignition probability
    if humid_col in df.columns:
        humidity = df[humid_col].fillna(50)
        # Sigmoid centered around 40% RH (critical threshold)
        humid_score = 1 / (1 + np.exp((humidity - 40) / 15))
    
    # Precipitation score: Less precip = higher ignition probability
    if precip_col in df.columns:
        precip = df[precip_col].fillna(0)
        # Exponential decay: more rain rapidly decreases ignition probability
        precip_score = np.exp(-precip / (precip.median() + 0.1))
    
    # Combined ignition threshold using weighted geometric mean
    # Geometric mean ensures all factors must be favorable
    result['ignition_soft_threshold'] = (
        (temp_score ** 0.4) * 
        (humid_score ** 0.35) * 
        (precip_score ** 0.25)
    ).clip(0, 1)
    
    return result


print("✓ Meteorological feature functions defined")


✓ Meteorological feature functions defined


In [8]:
# ============================================
# TERRAIN FEATURE TRANSFORMATIONS
# ============================================

def normalize_terrain_features(df):
    """
    Normalize and transform terrain features.
    
    Features:
    - elevation: meters (robust scaling due to outliers)
    - slope: degrees (min-max scaling to [0, 1])
    - ruggedness: terrain roughness index (min-max)
    - curvature: concavity/convexity (standard scaling)
    - canyons: binary/float canyon indicator
    - distance_to_water: meters (log transform + min-max)
    """
    result = pd.DataFrame(index=df.index)
    
    # Elevation: Use robust scaling (handles outliers)
    if 'elevation' in df.columns:
        elev = df['elevation'].fillna(df['elevation'].median())
        # Robust scaling using IQR
        q25, q75 = elev.quantile(0.25), elev.quantile(0.75)
        iqr = q75 - q25
        if iqr > 0:
            result['terrain_elevation'] = (elev - elev.median()) / iqr
        else:
            result['terrain_elevation'] = 0.0
    else:
        result['terrain_elevation'] = 0.0
    
    # Slope: Degrees normalized to [0, 1]
    # Max theoretical slope is 90 degrees
    if 'slope' in df.columns:
        slope = df['slope'].fillna(0)
        result['terrain_slope'] = (slope / 90.0).clip(0, 1)
    else:
        result['terrain_slope'] = 0.0
    
    # Ruggedness: Min-max normalize
    if 'ruggedness' in df.columns:
        rugged = df['ruggedness'].fillna(0)
        rug_min, rug_max = rugged.min(), rugged.max()
        if rug_max > rug_min:
            result['terrain_ruggedness'] = (rugged - rug_min) / (rug_max - rug_min)
        else:
            result['terrain_ruggedness'] = 0.0
    else:
        result['terrain_ruggedness'] = 0.0
    
    # Curvature: Standard scaling (can be positive or negative)
    if 'curvature' in df.columns:
        curv = df['curvature'].fillna(0)
        curv_mean, curv_std = curv.mean(), curv.std()
        if curv_std > 0:
            result['terrain_curvature'] = (curv - curv_mean) / curv_std
        else:
            result['terrain_curvature'] = 0.0
    else:
        result['terrain_curvature'] = 0.0
    
    # Canyon indicator: Keep as float [0, 1]
    if 'canyons' in df.columns:
        result['terrain_canyon'] = df['canyons'].fillna(0).clip(0, 1)
    else:
        result['terrain_canyon'] = 0.0
    
    # Distance to water: Log transform then normalize
    # Log transform handles the wide range of distances
    water_col = 'distance_to_water_meters' if 'distance_to_water_meters' in df.columns else None
    if water_col and water_col in df.columns:
        dist = df[water_col].fillna(df[water_col].median())
        # Log transform (add 1 to handle zeros)
        log_dist = np.log1p(dist)
        log_min, log_max = log_dist.min(), log_dist.max()
        if log_max > log_min:
            result['terrain_water_distance'] = (log_dist - log_min) / (log_max - log_min)
        else:
            result['terrain_water_distance'] = 0.5
    else:
        result['terrain_water_distance'] = 0.5
    
    return result


print("✓ Terrain feature functions defined")


✓ Terrain feature functions defined


In [ ]:
# ============================================
# VEGETATION / FUEL FEATURE TRANSFORMATIONS
# ============================================

def encode_forest_types(df):
    """
    Create one-hot encoding for forest types.
    
    Consolidates MODIS IGBP classes into fire-relevant categories:
    - forest_conifer: Evergreen/Deciduous Needleleaf (high fire risk)
    - forest_broadleaf: Evergreen/Deciduous Broadleaf
    - forest_mixed: Mixed forests
    - shrubland: Open/Closed shrublands (high fire risk)
    - savanna: Savannas/Woody Savannas (very high fire risk)
    - grassland: Grasslands (high fire risk, fast spread)
    - wetland: Permanent wetlands (low fire risk)
    - cropland: Croplands/Mosaics
    - barren: Barren/Urban (very low fire risk)
    - water: Water bodies (no fire risk)
    """
    result = pd.DataFrame(index=df.index)
    
    # Define forest type column mappings
    forest_type_cols = {
        'forest_conifer': ['forest_type_evergreen_needleleaf_forest', 'forest_type_deciduous_needleleaf_forest'],
        'forest_broadleaf': ['forest_type_evergreen_broadleaf_forest', 'forest_type_deciduous_broadleaf_forest'],
        'forest_mixed': ['forest_type_mixed_forests'],
        'shrubland': ['forest_type_closed_shrublands', 'forest_type_open_shrublands'],
        'savanna': ['forest_type_savannas', 'forest_type_woody_savannas'],
        'grassland': ['forest_type_grasslands'],
        'wetland': ['forest_type_permanent_wetlands'],
        'cropland': ['forest_type_croplands', 'forest_type_cropland_natural_mosaic'],
        'barren': ['forest_type_barren', 'forest_type_urban_built_up', 'forest_type_snow_ice'],
        'water': ['forest_type_water']
    }
    
    # Create consolidated one-hot encoding
    for category, source_cols in forest_type_cols.items():
        # Sum values from source columns (handles one-hot where only one should be 1)
        category_sum = pd.Series(0, index=df.index)
        for col in source_cols:
            if col in df.columns:
                category_sum = category_sum + df[col].fillna(0)
        result[f'veg_{category}'] = (category_sum > 0).astype(float)
    
    # If no forest type columns found, create defaults based on available data
    if not any(col.startswith('forest_type_') for col in df.columns):
        # Check for raw landcover class
        if 'landcover_class_raw' in df.columns:
            lc = df['landcover_class_raw'].fillna(255)
            result['veg_forest_conifer'] = lc.isin([1, 3]).astype(float)
            result['veg_forest_broadleaf'] = lc.isin([2, 4]).astype(float)
            result['veg_forest_mixed'] = (lc == 5).astype(float)
            result['veg_shrubland'] = lc.isin([6, 7]).astype(float)
            result['veg_savanna'] = lc.isin([8, 9]).astype(float)
            result['veg_grassland'] = (lc == 10).astype(float)
            result['veg_wetland'] = (lc == 11).astype(float)
            result['veg_cropland'] = lc.isin([12, 14]).astype(float)
            result['veg_barren'] = lc.isin([13, 15, 16]).astype(float)
            result['veg_water'] = (lc == 0).astype(float)
        else:
            # Default to unknown/mixed
            for category in forest_type_cols.keys():
                result[f'veg_{category}'] = 0.0
            result['veg_forest_mixed'] = 1.0  # Default assumption
    
    return result


def encode_fuel_layers(df):
    """
    Create encoding for fuel layers/load.
    
    Features:
    - fuel_load: Overall fuel load (continuous [0, 1])
    - fuel_model_category: Simplified fuel model (1-13 -> 4 categories)
    - canopy_metrics: Height, cover, surface/crown fuel
    """
    result = pd.DataFrame(index=df.index)
    
    # Fuel load (continuous)
    if 'fuel_load' in df.columns:
        result['fuel_load'] = df['fuel_load'].fillna(0.5).clip(0, 1)
    else:
        result['fuel_load'] = 0.5
    
    # Fuel model categories (simplify 13 models to 4 categories)
    # Model 1-3: Grass (light fuels, fast spread)
    # Model 4-7: Shrub (medium fuels)
    # Model 8-10: Timber understory (heavy fuels)
    # Model 11-13: Timber crown (very heavy fuels)
    
    fuel_model_raw = None
    if 'fuel_model_raw' in df.columns:
        fuel_model_raw = df['fuel_model_raw'].fillna(7)  # Default to shrub
    elif 'fuel_model' in df.columns:
        fuel_model_raw = df['fuel_model'].fillna(7)
    
    if fuel_model_raw is not None:
        result['fuel_category_grass'] = fuel_model_raw.isin([1, 2, 3]).astype(float)
        result['fuel_category_shrub'] = fuel_model_raw.isin([4, 5, 6, 7]).astype(float)
        result['fuel_category_timber_understory'] = fuel_model_raw.isin([8, 9, 10]).astype(float)
        result['fuel_category_timber_crown'] = fuel_model_raw.isin([11, 12, 13]).astype(float)
    else:
        # Default to shrub category
        result['fuel_category_grass'] = 0.0
        result['fuel_category_shrub'] = 1.0
        result['fuel_category_timber_understory'] = 0.0
        result['fuel_category_timber_crown'] = 0.0
    
    # Canopy metrics
    if 'canopy_height' in df.columns:
        # Normalize canopy height (assume max ~50m)
        result['fuel_canopy_height'] = (df['canopy_height'].fillna(5) / 50.0).clip(0, 1)
    else:
        result['fuel_canopy_height'] = 0.2
    
    if 'canopy_cover' in df.columns:
        result['fuel_canopy_cover'] = df['canopy_cover'].fillna(0.5).clip(0, 1)
    else:
        result['fuel_canopy_cover'] = 0.5
    
    # Surface and crown fuel loads
    if 'surface_fuel_load' in df.columns:
        result['fuel_surface'] = df['surface_fuel_load'].fillna(0.5).clip(0, 1)
    else:
        result['fuel_surface'] = 0.5
    
    if 'crown_fuel_load' in df.columns:
        result['fuel_crown'] = df['crown_fuel_load'].fillna(0.3).clip(0, 1)
    else:
        result['fuel_crown'] = 0.3
    
    return result


print("✓ Vegetation/Fuel feature functions defined")


✓ Vegetation/Fuel feature functions defined


In [10]:
# ============================================
# TEMPORAL / CIRCULAR FEATURE TRANSFORMATIONS
# ============================================

def encode_temporal_features(df):
    """
    Create circular encoding for temporal features.
    
    Season encoding (day-of-year):
    - season_sin: sin(2π * day_of_year / 365)
    - season_cos: cos(2π * day_of_year / 365)
    
    Time-of-day encoding (hour):
    - hour_sin: sin(2π * hour / 24)
    - hour_cos: cos(2π * hour / 24)
    
    Circular encoding preserves the cyclical nature of time.
    """
    result = pd.DataFrame(index=df.index)
    
    # Parse date and time
    if 'ACQ_DATE' in df.columns:
        # Convert to datetime if needed
        dates = pd.to_datetime(df['ACQ_DATE'], errors='coerce')
        day_of_year = dates.dt.dayofyear.fillna(182)  # Default to mid-year
    else:
        day_of_year = pd.Series([182] * len(df), index=df.index)
    
    # Parse time (ACQ_TIME is typically HHMM format as string or int)
    if 'ACQ_TIME' in df.columns:
        acq_time = df['ACQ_TIME'].astype(str).str.zfill(4)
        # Extract hour from HHMM format
        hour = pd.to_numeric(acq_time.str[:2], errors='coerce').fillna(12)
    else:
        hour = pd.Series([12] * len(df), index=df.index)  # Default to noon
    
    # Season encoding (day of year)
    day_angle = 2 * np.pi * day_of_year / 365.25
    result['temporal_season_sin'] = np.sin(day_angle)
    result['temporal_season_cos'] = np.cos(day_angle)
    
    # Time of day encoding (hour)
    hour_angle = 2 * np.pi * hour / 24.0
    result['temporal_hour_sin'] = np.sin(hour_angle)
    result['temporal_hour_cos'] = np.cos(hour_angle)
    
    return result


print("✓ Temporal feature functions defined")


✓ Temporal feature functions defined


In [11]:
# ============================================
# TARGET VARIABLE TRANSFORMATIONS
# ============================================

def create_target_variables(df):
    """
    Create target variables for the fire prediction model.
    
    Targets:
    - ignition_probability: Probability of fire ignition [0, 1]
      Derived from MODIS CONFIDENCE column (0-100 -> 0-1)
    
    - fire_start_time: Fire ignition timestamp
      Derived from ACQ_DATE + ACQ_TIME
      Also provide hours_since_midnight for numerical representation
    
    - fire_end_time: Fire cessation timestamp (or duration proxy)
      Since MODIS detects active fires, we estimate duration based on:
      - Fire radiative power (FRP): Higher FRP = longer duration
      - Fuel type: Different fuels burn at different rates
    """
    result = pd.DataFrame(index=df.index)
    
    # ===== Ignition Probability =====
    # MODIS CONFIDENCE is 0-100, scale to 0-1
    if 'CONFIDENCE' in df.columns:
        confidence = df['CONFIDENCE'].fillna(50)
        result['target_ignition_probability'] = (confidence / 100.0).clip(0, 1)
    else:
        result['target_ignition_probability'] = 0.5  # Default
    
    # ===== Fire Start Time =====
    # Combine date and time
    if 'ACQ_DATE' in df.columns and 'ACQ_TIME' in df.columns:
        dates = pd.to_datetime(df['ACQ_DATE'], errors='coerce')
        acq_time = df['ACQ_TIME'].astype(str).str.zfill(4)
        
        # Extract hours and minutes
        hours = pd.to_numeric(acq_time.str[:2], errors='coerce').fillna(12)
        minutes = pd.to_numeric(acq_time.str[2:4], errors='coerce').fillna(0)
        
        # Create timedelta and add to date
        time_delta = pd.to_timedelta(hours, unit='h') + pd.to_timedelta(minutes, unit='m')
        result['target_fire_start_datetime'] = dates + time_delta
        
        # Numeric representation: hours since midnight
        result['target_fire_start_hour'] = hours + minutes / 60.0
    else:
        result['target_fire_start_datetime'] = pd.NaT
        result['target_fire_start_hour'] = 12.0
    
    # ===== Fire End Time / Duration Proxy =====
    # Estimate duration based on FRP and fuel characteristics
    # This is a proxy since we don't have actual end times
    
    # Base duration estimate (hours) from FRP
    if 'FRP' in df.columns:
        frp = df['FRP'].fillna(df['FRP'].median() if 'FRP' in df.columns else 20)
        # Higher FRP typically indicates more intense fires that may burn longer
        # Scale: FRP 10 -> ~2 hours, FRP 100 -> ~8 hours, FRP 500+ -> ~24 hours
        base_duration = 2 + 6 * (np.log1p(frp) / np.log1p(500))
        base_duration = base_duration.clip(1, 72)  # 1 hour to 3 days max
    else:
        base_duration = pd.Series([4.0] * len(df), index=df.index)  # Default 4 hours
    
    result['target_fire_duration_hours'] = base_duration
    
    # Calculate end time
    if 'target_fire_start_datetime' in result.columns:
        duration_td = pd.to_timedelta(base_duration, unit='h')
        result['target_fire_end_datetime'] = result['target_fire_start_datetime'] + duration_td
    else:
        result['target_fire_end_datetime'] = pd.NaT
    
    return result


print("✓ Target variable functions defined")


✓ Target variable functions defined


## 3. Apply Transformations


In [12]:
# Apply all feature transformations
print("=" * 60)
print("APPLYING FEATURE TRANSFORMATIONS")
print("=" * 60)

# 1. Meteorological features
print("\n[1/5] Computing meteorological features...")
moisture_3day = compute_3day_moisture_indices(df)
print(f"  ✓ 3-day moisture indices: {list(moisture_3day.columns)}")

fuel_conditioning = compute_14day_fuel_conditioning_index(df)
print(f"  ✓ 14-day fuel conditioning: {list(fuel_conditioning.columns)}")

weather_extremes = compute_weighted_weather_extremes_12h(df)
print(f"  ✓ Weather extremes (12h): {list(weather_extremes.columns)}")

ignition_threshold = compute_ignition_soft_threshold(df)
print(f"  ✓ Ignition threshold: {list(ignition_threshold.columns)}")

# 2. Terrain features
print("\n[2/5] Normalizing terrain features...")
terrain_features = normalize_terrain_features(df)
print(f"  ✓ Terrain features: {list(terrain_features.columns)}")

# 3. Vegetation features
print("\n[3/5] Encoding vegetation features...")
forest_types = encode_forest_types(df)
print(f"  ✓ Forest types: {list(forest_types.columns)}")

fuel_layers = encode_fuel_layers(df)
print(f"  ✓ Fuel layers: {list(fuel_layers.columns)}")

# 4. Temporal features
print("\n[4/5] Encoding temporal features...")
temporal_features = encode_temporal_features(df)
print(f"  ✓ Temporal features: {list(temporal_features.columns)}")

# 5. Target variables
print("\n[5/5] Creating target variables...")
targets = create_target_variables(df)
print(f"  ✓ Target variables: {list(targets.columns)}")

print("\n" + "=" * 60)
print("✓ All transformations complete!")


APPLYING FEATURE TRANSFORMATIONS

[1/5] Computing meteorological features...
  ✓ 3-day moisture indices: ['moisture_wetness_3day', 'moisture_dryness_3day', 'moisture_humidity_3day']
  ✓ 14-day fuel conditioning: ['fuel_conditioning_14day']
  ✓ Weather extremes (12h): ['weather_extreme_wind_12h', 'weather_extreme_precip_12h']
  ✓ Ignition threshold: ['ignition_soft_threshold']

[2/5] Normalizing terrain features...
  ✓ Terrain features: ['terrain_elevation', 'terrain_slope', 'terrain_ruggedness', 'terrain_curvature', 'terrain_canyon', 'terrain_water_distance']

[3/5] Encoding vegetation features...


AttributeError: 'bool' object has no attribute 'astype'

In [ ]:
# Combine all features into final dataframe
print("\nCombining all features...")

# Features (input to model)
features_df = pd.concat([
    # Meteorological features
    moisture_3day,
    fuel_conditioning,
    weather_extremes,
    ignition_threshold,
    
    # Terrain features
    terrain_features,
    
    # Vegetation features
    forest_types,
    fuel_layers,
    
    # Temporal features
    temporal_features
], axis=1)

# Targets (output from model)
targets_df = targets.copy()

# Add location metadata for reference (not used in model)
metadata_cols = ['LATITUDE', 'LONGITUDE', 'ACQ_DATE', 'ACQ_TIME']
metadata_df = df[[c for c in metadata_cols if c in df.columns]].copy()

# Full dataset
full_df = pd.concat([metadata_df, features_df, targets_df], axis=1)

print(f"\n✓ Features: {len(features_df.columns)} columns")
print(f"✓ Targets: {len(targets_df.columns)} columns")
print(f"✓ Metadata: {len(metadata_df.columns)} columns")
print(f"✓ Total: {len(full_df.columns)} columns, {len(full_df)} samples")


## 4. Data Validation and Quality Check


In [ ]:
# Validate feature ranges and data quality
print("=" * 60)
print("DATA VALIDATION")
print("=" * 60)

# Check for missing values
print("\n--- Missing Values ---")
missing = features_df.isnull().sum()
if missing.sum() > 0:
    print("Features with missing values:")
    print(missing[missing > 0])
else:
    print("✓ No missing values in features")

# Check feature ranges
print("\n--- Feature Value Ranges ---")
feature_stats = features_df.describe().T
feature_stats['range'] = feature_stats['max'] - feature_stats['min']
print(feature_stats[['min', 'max', 'mean', 'std', 'range']])


In [ ]:
# Check for infinite values
print("\n--- Infinite Values Check ---")
inf_counts = np.isinf(features_df.select_dtypes(include=[np.number])).sum()
if inf_counts.sum() > 0:
    print("⚠ Features with infinite values:")
    print(inf_counts[inf_counts > 0])
else:
    print("✓ No infinite values")

# Replace any infinite values with NaN, then fill
features_df = features_df.replace([np.inf, -np.inf], np.nan)
features_df = features_df.fillna(features_df.median())

# Validate target variables
print("\n--- Target Variable Validation ---")

# Ignition probability should be [0, 1]
if 'target_ignition_probability' in targets_df.columns:
    prob = targets_df['target_ignition_probability']
    print(f"Ignition probability: min={prob.min():.3f}, max={prob.max():.3f}, mean={prob.mean():.3f}")
    if prob.min() >= 0 and prob.max() <= 1:
        print("  ✓ Within valid range [0, 1]")
    else:
        print("  ⚠ Out of expected range!")

# Fire duration should be positive
if 'target_fire_duration_hours' in targets_df.columns:
    dur = targets_df['target_fire_duration_hours']
    print(f"Fire duration (hours): min={dur.min():.1f}, max={dur.max():.1f}, mean={dur.mean():.1f}")

print("\n✓ Data validation complete!")


## 5. Feature Summary by Category


In [ ]:
# Organize features by category
print("=" * 60)
print("ML-READY FEATURE SUMMARY")
print("=" * 60)

feature_categories = {
    'Meteorological (7 features)': [
        col for col in features_df.columns 
        if col.startswith(('moisture_', 'fuel_conditioning_', 'weather_extreme_', 'ignition_'))
    ],
    'Terrain (6 features)': [
        col for col in features_df.columns 
        if col.startswith('terrain_')
    ],
    'Vegetation (10+ features)': [
        col for col in features_df.columns 
        if col.startswith('veg_')
    ],
    'Fuel (8+ features)': [
        col for col in features_df.columns 
        if col.startswith('fuel_')
    ],
    'Temporal (4 features)': [
        col for col in features_df.columns 
        if col.startswith('temporal_')
    ]
}

for category, cols in feature_categories.items():
    print(f"\n{category}:")
    for col in cols:
        stats = features_df[col].describe()
        print(f"  • {col}: [{stats['min']:.3f}, {stats['max']:.3f}] μ={stats['mean']:.3f}")

print(f"\n" + "=" * 60)
print(f"TOTAL FEATURES: {len(features_df.columns)}")
print("=" * 60)


## 6. Save ML-Ready Dataset


In [ ]:
# Save datasets
print("=" * 60)
print("SAVING ML-READY DATASETS")
print("=" * 60)

# Save features only
features_df.to_parquet(OUTPUT_FEATURES, index=False)
print(f"\n✓ Features saved to: {OUTPUT_FEATURES}")
print(f"  Shape: {features_df.shape}")
print(f"  Size: {OUTPUT_FEATURES.stat().st_size / 1024:.1f} KB")

# Save targets only
targets_df.to_parquet(OUTPUT_TARGETS, index=False)
print(f"\n✓ Targets saved to: {OUTPUT_TARGETS}")
print(f"  Shape: {targets_df.shape}")
print(f"  Size: {OUTPUT_TARGETS.stat().st_size / 1024:.1f} KB")

# Save full dataset (features + targets + metadata)
full_df = pd.concat([metadata_df, features_df, targets_df], axis=1)
full_df.to_parquet(OUTPUT_FULL, index=False)
print(f"\n✓ Full dataset saved to: {OUTPUT_FULL}")
print(f"  Shape: {full_df.shape}")
print(f"  Size: {OUTPUT_FULL.stat().st_size / 1024:.1f} KB")


In [ ]:
# Create feature manifest for documentation
import json

print("\n" + "=" * 60)
print("FEATURE MANIFEST")
print("=" * 60)

manifest = {
    'total_samples': len(features_df),
    'total_features': len(features_df.columns),
    'total_targets': len([c for c in targets_df.columns if targets_df[c].dtype in [np.float64, np.int64]]),
    'features': {
        'meteorological': {
            'count': 7,
            'columns': [
                'moisture_dryness_3day',
                'moisture_wetness_3day', 
                'moisture_humidity_3day',
                'fuel_conditioning_14day',
                'weather_extreme_wind_12h',
                'weather_extreme_precip_12h',
                'ignition_soft_threshold'
            ]
        },
        'terrain': {
            'count': 6,
            'columns': [
                'terrain_elevation',
                'terrain_slope',
                'terrain_ruggedness',
                'terrain_curvature',
                'terrain_canyon',
                'terrain_water_distance'
            ]
        },
        'vegetation': {
            'count': len([c for c in features_df.columns if c.startswith('veg_')]),
            'columns': [c for c in features_df.columns if c.startswith('veg_')]
        },
        'fuel': {
            'count': len([c for c in features_df.columns if c.startswith('fuel_')]),
            'columns': [c for c in features_df.columns if c.startswith('fuel_')]
        },
        'temporal': {
            'count': 4,
            'columns': [
                'temporal_season_sin',
                'temporal_season_cos',
                'temporal_hour_sin',
                'temporal_hour_cos'
            ]
        }
    },
    'targets': {
        'ignition_probability': 'float [0, 1] - Probability of fire ignition',
        'fire_start_hour': 'float [0, 24) - Hour of fire detection',
        'fire_duration_hours': 'float [1, 72] - Estimated fire duration in hours'
    }
}

manifest_path = ML_READY_DIR / 'feature_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"\n✓ Feature manifest saved to: {manifest_path}")
print(json.dumps(manifest, indent=2))


## 7. Final Summary


In [ ]:
print("\n" + "=" * 70)
print("                    ML-READY DATASET COMPLETE")
print("=" * 70)

veg_count = len([c for c in features_df.columns if c.startswith('veg_')])
fuel_count = len([c for c in features_df.columns if c.startswith('fuel_')])
target_count = len([c for c in targets_df.columns if targets_df[c].dtype in [np.float64, np.int64]])

print(f"""
📊 Dataset Statistics:
   • Total samples: {len(features_df):,}
   • Total features: {len(features_df.columns)}
   • Total targets: {target_count}

📁 Output Files:
   • Features: {OUTPUT_FEATURES}
   • Targets: {OUTPUT_TARGETS}
   • Full dataset: {OUTPUT_FULL}
   • Manifest: {manifest_path}

🔧 Feature Categories:
   • Meteorological & Fuel Conditioning: 7 features
     - 3-day moisture indices (×3)
     - 14-day fuel conditioning index
     - Weather extremes (×2)
     - Ignition soft-threshold
   
   • Geospatial / Terrain: 6 features
     - Elevation, Slope, Ruggedness, Curvature, Canyon, Water distance
   
   • Vegetation / Forest Types: {veg_count} features (one-hot)
   
   • Fuel Layers: {fuel_count} features
   
   • Temporal / Circular: 4 features
     - Season (sin/cos), Time-of-day (sin/cos)

🎯 Target Variables:
   • target_ignition_probability: Fire ignition likelihood [0, 1]
   • target_fire_start_hour: Predicted start time (hour of day)
   • target_fire_duration_hours: Estimated fire duration

✅ Dataset is ready for machine learning!
""")


## 6.5. Create Train/Validation/Test Split Indices

This section creates flexible split indices saved as JSON, allowing easy changes to split ratios without regenerating the full dataset.


In [ ]:
# Create train/validation/test split indices
from sklearn.model_selection import train_test_split
import json

print("=" * 60)
print("CREATING TRAIN/VALIDATION/TEST SPLITS")
print("=" * 60)

# Configuration for splits (can be easily modified)
TRAIN_SIZE = 0.70  # 70% for training
VAL_SIZE = 0.15    # 15% for validation
TEST_SIZE = 0.15   # 15% for testing

# Get total number of samples
n_samples = len(features_df)
print(f"\nTotal samples: {n_samples}")

# Create indices array (0 to n_samples-1)
indices = np.arange(n_samples)

# First split: separate test set from train+val
# This ensures test set is never seen during training/validation
train_val_indices, test_indices = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=42,  # For reproducibility
    shuffle=True
)

# Second split: separate train and validation from remaining data
# Adjust val_size to account for the fact we're splitting train_val_indices
adjusted_val_size = VAL_SIZE / (1 - TEST_SIZE)
train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=adjusted_val_size,
    random_state=42,  # For reproducibility
    shuffle=True
)

# Convert to lists for JSON serialization
split_indices = {
    'train': train_indices.tolist(),
    'val': val_indices.tolist(),
    'test': test_indices.tolist()
}

# Save split indices to JSON file
with open(OUTPUT_SPLIT_INDICES, 'w') as f:
    json.dump(split_indices, f, indent=2)

print(f"\n✓ Split indices saved to: {OUTPUT_SPLIT_INDICES}")
print(f"  Train samples: {len(split_indices['train'])} ({len(split_indices['train'])/n_samples*100:.1f}%)")
print(f"  Validation samples: {len(split_indices['val'])} ({len(split_indices['val'])/n_samples*100:.1f}%)")
print(f"  Test samples: {len(split_indices['test'])} ({len(split_indices['test'])/n_samples*100:.1f}%)")
print(f"  Total: {sum(len(v) for v in split_indices.values())}")

# Verify no overlap
train_set = set(split_indices['train'])
val_set = set(split_indices['val'])
test_set = set(split_indices['test'])

assert len(train_set & val_set) == 0, "Train and validation sets overlap!"
assert len(train_set & test_set) == 0, "Train and test sets overlap!"
assert len(val_set & test_set) == 0, "Validation and test sets overlap!"
print("\n✓ Split validation: No overlaps detected - splits are clean!")

print("\n--- How to use split_indices.json ---")
print("To load a specific split in your training code:")
print("  1. Load features: features_df = pd.read_parquet('features.parquet')")
print("  2. Load targets: targets_df = pd.read_parquet('targets.parquet')")
print("  3. Load indices: with open('split_indices.json') as f: splits = json.load(f)")
print("  4. Get train split: train_features = features_df.iloc[splits['train']]")
print("  5. Get train targets: train_targets = targets_df.iloc[splits['train']]")


In [ ]:
# Quick verification: Load and check the saved dataset
print("\n--- Verification: Loading saved dataset ---")
loaded_features = pd.read_parquet(OUTPUT_FEATURES)
loaded_targets = pd.read_parquet(OUTPUT_TARGETS)

print(f"Features loaded: {loaded_features.shape}")
print(f"Targets loaded: {loaded_targets.shape}")
print("\n✓ All files verified successfully!")

# Display sample of final data
print("\n--- Sample of ML-Ready Features (first 3 rows) ---")
loaded_features.head(3)
